# 💬 Генерация текстов: от Т9 до чат-бота

> Сначала: **Файл → Сохранить копию на Диске**. Затем: **Среда выполнения → Сменить среду → GPU**.

Как телефон подсказывает следующее слово? Как чат-бот придумывает ответ? Идея одна и та же: **угадать следующее слово, потом ещё одно, и ещё**.

Сегодня:
1. **Т9 своими руками:** считаем, какое слово обычно идёт за каким, и пишем «бред по статистике».
2. **Чат-бот Seq2Seq:** одна LSTM читает вопрос, вторая пишет ответ.

Данные: почти 12 тысяч пар «реплика → ответ» из фильмов (датасет из LXP).

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, диалоги, проверки { display-mode: "form" }
import os, re, random, urllib.request
from collections import Counter, defaultdict
import numpy as np, matplotlib.pyplot as plt, yaml
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

if not os.path.exists("dialogs.yml"):
    urllib.request.urlretrieve("https://drive.usercontent.google.com/download?id=1h7cVEDJ3BR_4xgtRSCEtZJOiHcVHZLJX&export=download&confirm=t", "dialogs.yml")
random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

def tokenize(text):
    return re.findall(r"[а-яёa-z]+", str(text).lower())

def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True

print("✅ Готово. Устройство:", device)

In [ ]:
data = yaml.safe_load(open("dialogs.yml", encoding="utf-8-sig"))["разговоры"]
pairs = [(tokenize(d[0]), tokenize(" ".join(map(str, d[1:])))) for d in data if len(d) >= 2]
pairs = [(q, a) for q, a in pairs if 0 < len(q) <= 10 and 0 < len(a) <= 10]   # короткие фразы
print("Пар:", len(pairs))
for q, a in random.sample(pairs, 5):
    print(" ".join(q), "  →  ", " ".join(a))

---
# Часть 1. Т9 своими руками

Посчитаем для каждого слова, **какие слова идут после него и сколько раз**. Например, после «как» часто идёт «дела», «ты», «же».

```
follow["как"] = Counter({"же": 120, "ты": 95, "дела": 40, ...})
```

### ✏️ Задание 1. Заполните `follow`: для каждой пары соседних слов увеличьте счётчик

In [ ]:
follow = defaultdict(Counter)
for q, a in pairs:
    for phrase in (q, a):
        for w1, w2 in zip(phrase, phrase[1:]):   # соседние слова: (w1, w2)
            pass  # после w1 встретилось w2: прибавьте 1

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('есть слова', lambda: len(follow) > 1000, 'Внутри цикла увеличивайте follow[w1][w2]'),
    ('всё посчитано', lambda: sum(sum(c.values()) for c in follow.values()) == sum(len(p) - 1 for q, a in pairs for p in (q, a)), 'Каждая пара соседей добавляет ровно 1'),
], 1):
    for w in ["как", "я", "где"]:
        print(w, "→", follow[w].most_common(5))

<details><summary>💡 Подсказка</summary>

`follow[w1]` — это Counter. К его элементу можно прибавить 1, как к обычному числу.
</details>

### ✏️ Задание 2. Жадный Т9: `continue_greedy(word, n)` каждый раз берёт самое частое следующее слово

In [ ]:
def continue_greedy(word, n=8):
    words = [word]
    for _ in range(n):
        if not follow[word]:
            break
        word = None  # самое частое слово после word
        words.append(word)
    return words

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('начало', lambda: continue_greedy("как")[0] == "как", 'Список начинается с исходного слова'),
    ('шаг', lambda: continue_greedy("как", 1)[1] == follow["как"].most_common(1)[0][0], 'most_common(1) возвращает список [(слово, сколько)]'),
], 2):
    for w in ["я", "ты", "что"]:
        print(" ".join(continue_greedy(w)))

<details><summary>💡 Подсказка</summary>

`most_common(1)` даёт `[(слово, число)]`. Достаньте из него слово.
</details>

Заметили? Жадный Т9 быстро **зацикливается**: «я не знаю что я не знаю что…». Он всегда выбирает одно и то же.

Решение — выбирать **случайно, но с учётом частоты**. А **температура** решает, насколько смело:

| Температура | Что происходит |
|---|---|
| 0.2 | почти всегда самое частое слово: скучно, повторы |
| 1.0 | честно по частотам |
| 2.0 | редкие слова выпадают часто: смешно и бессвязно |

In [ ]:
#@title ⚙️ Запустите: функция случайного выбора с температурой { display-mode: "form" }
def sample(counter, temperature=1.0):
    """Выбирает слово из Counter случайно: чем чаще слово, тем больше шанс.
    temperature < 1 — смелее берём частые слова, > 1 — чаще редкие."""
    words = list(counter)
    p = np.array([counter[w] for w in words], dtype=float) ** (1 / temperature)
    return np.random.choice(words, p=p / p.sum())

print("✅ Готово")

### ✏️ Задание 3. Случайный Т9: `continue_random(word, n, temperature)` выбирает слово функцией `sample`

In [ ]:
def continue_random(word, n=8, temperature=1.0):
    words = [word]
    for _ in range(n):
        if not follow[word]:
            break
        word = None  # sample(счётчик слов после word, temperature)
        words.append(word)
    return words

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('длина', lambda: len(continue_random("я", 5)) <= 6, 'Не больше n новых слов'),
    ('строки', lambda: all(isinstance(w, str) for w in continue_random("я")), 'sample возвращает слово'),
    ('разные ответы', lambda: len({" ".join(continue_random("я", 6, 1.5)) for _ in range(10)}) > 1, 'Ответ должен меняться от запуска к запуску'),
], 3):
    for t in [0.2, 1.0, 2.0]:
        print(f"t = {t}:", " ".join(continue_random("я", 10, t)))

<details><summary>💡 Подсказка</summary>

Передайте в `sample` счётчик `follow[word]` и температуру.
</details>

Т9 помнит только **одно** предыдущее слово, поэтому фразы не связаны по смыслу. И он не умеет **отвечать**: ему всё равно, какой был вопрос.

Нужна сеть, которая сначала **прочитает весь вопрос**, а потом **напишет ответ**. Это Seq2Seq.

---
# Часть 2. Готовим данные для Seq2Seq

```
вопрос:  «как дела»  →  [кодировщик LSTM]  →  смысл (h, c)
                                                  ↓
ответ:   <BOS> → нормально → <EOS>   ←  [декодировщик LSTM]
```

Служебные слова: `<PAD>` — пустое место, `<UNK>` — незнакомое слово, `<BOS>` — «начни ответ», `<EOS>` — «ответ закончен».

In [ ]:
counts = Counter(w for q, a in pairs for w in q + a)
itos = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"] + [w for w, c in counts.most_common() if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
PAD, UNK, BOS, EOS = 0, 1, 2, 3
VOCAB = len(itos)

def encode(words):
    return [stoi.get(w, UNK) for w in words]

print("Размер словаря:", VOCAB)
print(encode(["как", "дела", "абырвалг"]))

**Как учится декодировщик.** На каждом шаге он получает **правильное прошлое слово** и должен угадать следующее. Поэтому вход и выход сдвинуты на одно слово:

| Шаг | 1 | 2 | 3 |
|---|---|---|---|
| вход декодировщика | `<BOS>` | ждёт | вас |
| правильный выход | ждёт | вас | `<EOS>` |

### ✏️ Задание 4. Напишите `decoder_io(ids)`: вход = BOS + ответ, выход = ответ + EOS

In [ ]:
def decoder_io(ids):
    dec_in = None
    dec_out = None
    return dec_in, dec_out

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('вход', lambda: decoder_io([7, 8])[0] == [BOS, 7, 8], 'В начале входа стоит BOS'),
    ('выход', lambda: decoder_io([7, 8])[1] == [7, 8, EOS], 'В конце выхода стоит EOS'),
    ('сдвиг', lambda: decoder_io([5])[0][1:] == decoder_io([5])[1][:-1], 'Выход — это вход, сдвинутый на одно слово'),
], 4):
    pass

<details><summary>💡 Подсказка</summary>

Списки складываются плюсом: `[1] + [2, 3]`.
</details>

In [ ]:
L = 11   # 10 слов + служебное

def pad_left(ids):    # вопрос: нули слева, чтобы кодировщик закончил на настоящем слове
    return [PAD] * (L - len(ids)) + ids

def pad_right(ids):   # ответ: нули справа, их не штрафуем
    return ids + [PAD] * (L - len(ids))

Q = torch.tensor([pad_left(encode(q)) for q, a in pairs])
io = [decoder_io(encode(a)) for q, a in pairs]
DI = torch.tensor([pad_right(i) for i, o in io])
DO = torch.tensor([pad_right(o) for i, o in io])
print(Q.shape, DI.shape, DO.shape)
print("вопрос:", Q[0].tolist())
print("вход:  ", DI[0].tolist())
print("выход: ", DO[0].tolist())

---
# Часть 3. Модель

Две LSTM и общий словарь эмбеддингов. Главное — **передать память кодировщика `(h, c)` декодировщику**: так декодировщик «знает», на что отвечает.

### ✏️ Задание 5. Соберите `Seq2Seq`: прогоните вопрос через кодировщик и передайте его память декодировщику

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab, emb=128, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.encoder = nn.LSTM(emb, hidden, batch_first=True)
        self.decoder = nn.LSTM(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)    # оценка каждого слова словаря

    def forward(self, q, dec_in):
        _, (h, c) = None, (None, None)  # кодировщик читает self.emb(q)
        out, _ = None, None  # декодировщик читает self.emb(dec_in) и стартует с (h, c)
        return self.head(out)

model = Seq2Seq(VOCAB).to(device)

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('форма', lambda: model(Q[:2].to(device), DI[:2].to(device)).shape == (2, L, VOCAB), 'На каждое слово ответа — оценки всех слов словаря'),
    ('память передана', lambda: not torch.allclose(model(Q[:1].to(device), DI[:1].to(device)), model(Q[1:2].to(device), DI[:1].to(device))), 'Декодировщику нужен второй аргумент (h, c), иначе вопрос не влияет на ответ'),
], 5):
    pass

<details><summary>💡 Подсказка</summary>

LSTM возвращает `out, (h, c)`. Второй аргумент LSTM — начальная память.
</details>

---
# Часть 4. Как сеть пишет ответ

Обучение — это одно, а **ответ** сеть пишет по слову, как Т9:
1. кодировщик читает вопрос → память;
2. даём декодировщику `<BOS>`, он выдаёт оценки слов;
3. выбираем слово (самое вероятное или случайно с температурой);
4. подаём это слово обратно и повторяем, пока не выпадет `<EOS>`.

### ✏️ Задание 6. Допишите `reply`: выбор самого вероятного слова и остановку на EOS

In [ ]:
def reply(text, temperature=0.0, max_len=10):
    model.eval()
    q = torch.tensor([pad_left(encode(tokenize(text))[-10:])]).to(device)
    with torch.no_grad():
        _, state = model.encoder(model.emb(q))
        word, answer = BOS, []
        for _ in range(max_len):
            out, state = model.decoder(model.emb(torch.tensor([[word]], device=device)), state)
            logits = model.head(out[0, -1])
            logits[UNK] = -1e9                     # не отвечаем «незнакомым словом»
            if temperature == 0:
                word = None  # номер самого вероятного слова: argmax, затем int
            else:
                probs = torch.softmax(logits / temperature, dim=0)
                word = int(torch.multinomial(probs, 1))
            pass  # если word равно EOS — выйдите из цикла
            answer.append(itos[word])
    return " ".join(answer)

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('строка', lambda: isinstance(reply("привет"), str), 'Функция возвращает строку'),
    ('длина', lambda: len(reply("привет", 1.0).split()) <= 10, 'Не больше max_len слов'),
    ('без служебных', lambda: "<EOS>" not in reply("как дела", 1.0), 'При EOS выходите из цикла, не добавляя его в ответ'),
], 6):
    print("До обучения сеть несёт чепуху:")
    for t in ["привет", "как дела", "ты кто"]:
        print(" ", t, "→", reply(t, 1.0))

<details><summary>💡 Подсказка</summary>

`logits.argmax()` даёт тензор с номером, `int(...)` превращает его в число.
</details>

In [ ]:
#@title ⚙️ Запустите: функция обучения (читать не обязательно) { display-mode: "form" }
def train(model, epochs, lr=0.001):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)   # на паддинге не штрафуем
    loader = DataLoader(TensorDataset(Q, DI, DO), batch_size=64, shuffle=True)
    losses = []
    for ep in range(1, epochs + 1):
        model.train(); total = 0
        for q, di, do in loader:
            q, di, do = q.to(device), di.to(device), do.to(device)
            logits = model(q, di)
            loss = loss_fn(logits.reshape(-1, logits.shape[-1]), do.reshape(-1))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total += loss.item() * len(q)
        losses.append(total / len(Q))
        if ep % 5 == 0 or ep == 1:
            print(f"эпоха {ep:2d}: loss {losses[-1]:.3f} | «привет» → {reply('привет', 0)}")
    return losses

print("✅ Готово")

### ✏️ Задание 7. Обучите чат-бота не меньше 20 эпох и посмотрите, как меняется ответ на «привет»

In [ ]:
losses = train(model, epochs=None)

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('эпохи', lambda: len(losses) >= 20, 'Нужно минимум 20 эпох'),
    ('учится', lambda: losses[-1] < losses[0] * 0.8, 'Loss должен заметно падать. Проверьте модель'),
], 7):
    plt.plot(losses); plt.title("loss"); plt.xlabel("эпоха"); plt.show()

<details><summary>💡 Подсказка</summary>

Около 30 эпох достаточно. С GPU это пара минут.
</details>

---
# Часть 5. Поговорим с ботом

In [ ]:
for q in ["привет", "как дела", "ты кто", "где мы", "что ты делаешь", "пойдём домой"]:
    print(f"{q:16} | строго: {reply(q, 0):25} | творчески: {reply(q, 0.8)}")

### ✏️ Задание 8. Задайте боту свои вопросы: `my_questions` — список минимум из трёх строк

In [ ]:
my_questions = None
my_answers = [reply(q, 0.7) for q in my_questions]

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('вопросы', lambda: isinstance(my_questions, list) and len(my_questions) >= 3 and all(isinstance(q, str) for q in my_questions), 'Список минимум из трёх строк'),
    ('ответы', lambda: len(my_answers) == len(my_questions), 'Запустите ячейку целиком'),
], 8):
    for q, a in zip(my_questions, my_answers):
        print("Вы: ", q)
        print("Бот:", a, "\n")

<details><summary>💡 Подсказка</summary>

Пишите короткие разговорные фразы: бот учился на диалогах из фильмов.
</details>

In [ ]:
# Живой чат: раскомментируйте и запустите. Пустая строка — выход.
# while True:
#     q = input("Вы: ")
#     if not q:
#         break
#     print("Бот:", reply(q, 0.7))

**Почему бот часто отвечает «я не знаю» или «да»?** Это самые частые ответы в данных: когда сеть не уверена, она выбирает то, что чаще всего было правильно. Температура добавляет разнообразия, но и чепухи.

**Как сделать лучше:** больше данных, больше эпох, **внимание** (attention), а в пределе — трансформер. ChatGPT делает то же, что наш бот, — угадывает следующее слово, — только обучен на триллионах слов.

---
# Часть 6. Задание LXP: бот на своём датасете

Нужно обучить такую же сеть на **своих** парах «вопрос → ответ» (лучше около 10 тысяч):
1. Найдите или соберите диалоги: датасет из интернета, свои переписки, разговорник для переводчика.
2. Очистите от смайликов и мусора (`tokenize` оставляет только буквы).
3. Сохраните в `my_dialogs.csv` со столбцами `question` и `answer`.
4. Раскомментируйте ячейку ниже, затем **перезапустите ноутбук с Части 2** (Среда выполнения → Выполнить ячейки после текущей).

In [ ]:
# import pandas as pd
# my = pd.read_csv("my_dialogs.csv").dropna()
# pairs = [(tokenize(q), tokenize(a)) for q, a in zip(my["question"], my["answer"])]
# pairs = [(q, a) for q, a in pairs if 0 < len(q) <= 10 and 0 < len(a) <= 10]
# print("Пар:", len(pairs))

## 🏁 Итог
* Генерация = **угадать следующее слово** и повторить.
* Жадный выбор зацикливается, **температура** добавляет разнообразие.
* **Seq2Seq:** кодировщик сжимает вопрос в память `(h, c)`, декодировщик пишет ответ с `<BOS>` до `<EOS>`.
* При обучении декодировщик видит правильное прошлое слово, при ответе — своё.